# Schedule Market Calendar Gate v1

目标：把交易日、半日市、休市判断沉到 `src/schedule`，让数据更新、模型刷新、dashboard export 和 paper/sim 执行在进入任务前先有稳定 gate。

In [1]:
from pathlib import Path
import json
import sys
import tempfile

import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import (
    DataUpdatePipelineConfig,
    ModelDashboardPipelineConfig,
    PaperExecutionPipelineConfig,
    ScheduleJob,
    classify_market_calendar_day,
    run_data_update_once,
    run_job_once,
    run_model_dashboard_refresh_once,
    run_paper_execution_once,
)

OUTPUT_DIR = REPO_ROOT / "research" / "schedule_market_calendar_gate_v1"
trading_calendar = ("2026-01-02", "2026-01-05", "2026-01-06")
half_days = ("2026-01-02",)
holiday_reasons = {"2026-01-01": "New Year's Day"}

decisions = [
    classify_market_calendar_day("2026-01-01", trading_calendar=trading_calendar, half_days=half_days, holiday_reasons=holiday_reasons),
    classify_market_calendar_day("2026-01-02", trading_calendar=trading_calendar, half_days=half_days, holiday_reasons=holiday_reasons),
    classify_market_calendar_day("2026-01-05", trading_calendar=trading_calendar, half_days=half_days, holiday_reasons=holiday_reasons),
]

closed_decision = decisions[0]
events: list[str] = []
calls = {"count": 0}
with tempfile.TemporaryDirectory() as raw_tmp:
    job = ScheduleJob(name="paper_execution", task="paper_sim", cron="manual")
    closed_result = run_job_once(
        job,
        task=lambda: calls.__setitem__("count", calls["count"] + 1) or {"should_not_run": True},
        lock_dir=Path(raw_tmp),
        notify=events.append,
        market_calendar_decision=closed_decision,
    )
    closed_lock_exists = (Path(raw_tmp) / "paper_execution.lock").exists()

notification_events: list[str] = []
notification_calls = {"count": 0}
with tempfile.TemporaryDirectory() as raw_tmp:
    notified_skip_result = run_job_once(
        ScheduleJob(name="paper_execution", task="paper_sim", cron="manual"),
        task=lambda: notification_calls.__setitem__("count", notification_calls["count"] + 1) or {"should_not_run": True},
        lock_dir=Path(raw_tmp),
        notify=notification_events.append,
        market_calendar_decision=closed_decision,
        notify_skipped=True,
    )
    notified_skip_lock_exists = (Path(raw_tmp) / "paper_execution.lock").exists()

pipeline_events: list[str] = []
pipeline_calls = {"data_frame": 0, "data_calendar": 0, "matrix": 0, "dashboard": 0}
with tempfile.TemporaryDirectory() as raw_tmp:
    tmp_root = Path(raw_tmp)
    locks = tmp_root / "locks"

    data_config = DataUpdatePipelineConfig(
        name="daily_soxl_data_update",
        dataset_key="daily.soxl_unit",
        dataset_name="unit_soxl_daily",
        universe=("SOXL",),
        output_dir=tmp_root / "data" / "snapshot",
        registry_path=tmp_root / "data" / "data_registry.json",
        calendar_path="data/qlib/soxl/calendars/day.txt",
        source_paths=("provider:unit",),
        start="2026-01-01",
        end="2026-01-05",
        as_of="2026-01-05",
        notify_details=True,
        market_calendar_decision=closed_decision,
    )

    def frame_provider(_: DataUpdatePipelineConfig) -> pd.DataFrame:
        pipeline_calls["data_frame"] += 1
        return pd.DataFrame([{"date": "2026-01-02", "instrument": "SOXL", "close": 10.0}])

    def calendar_provider(_: DataUpdatePipelineConfig) -> pd.DatetimeIndex:
        pipeline_calls["data_calendar"] += 1
        return pd.DatetimeIndex(pd.to_datetime(["2026-01-02"]), name="date")

    data_result = run_data_update_once(
        data_config,
        lock_dir=locks,
        notify=pipeline_events.append,
        frame_provider=frame_provider,
        calendar_provider=calendar_provider,
    )

    model_config = ModelDashboardPipelineConfig(
        name="model_dashboard_refresh",
        card_dir=tmp_root / "matrix_card",
        dashboard_output_path=tmp_root / "dashboard" / "dashboard_export.json",
        start="2026-01-01",
        end="2026-06-09",
        train_end="2026-04-30",
        market_calendar_decision=closed_decision,
    )

    def matrix_runner(**kwargs):
        pipeline_calls["matrix"] += 1
        Path(kwargs["card_dir"]).mkdir(parents=True, exist_ok=True)
        return {"variant_count": 1}

    def dashboard_exporter(card_dir: Path, *, output_path: Path, top_n: int):
        pipeline_calls["dashboard"] += 1
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(json.dumps({"schema_version": "dashboard.model_matrix.v1"}), encoding="utf-8")
        return {"schema_version": "dashboard.model_matrix.v1"}

    model_result = run_model_dashboard_refresh_once(
        model_config,
        lock_dir=locks,
        notify=pipeline_events.append,
        matrix_runner=matrix_runner,
        dashboard_exporter=dashboard_exporter,
    )

    paper_config = PaperExecutionPipelineConfig(
        name="paper_execution",
        dashboard_export_path=tmp_root / "missing_dashboard_export.json",
        data_registry_path=tmp_root / "missing_data_registry.json",
        dataset_key="daily.soxl_unit",
        instrument="SOXL",
        intent_ledger_path=tmp_root / "runtime" / "intent_ledger.json",
        paper_execution_result_path=tmp_root / "runtime" / "paper_execution_result.json",
        notify_details=True,
        market_calendar_decision=closed_decision,
    )
    paper_result = run_paper_execution_once(paper_config, lock_dir=locks, notify=pipeline_events.append)

    pipeline_lock_created = any(locks.glob("*.lock")) if locks.exists() else False
    pipeline_artifact_created = any(
        path.exists()
        for path in (
            data_config.output_dir,
            model_config.card_dir,
            model_config.dashboard_output_path,
            paper_config.intent_ledger_path,
            paper_config.paper_execution_result_path,
        )
    )

pipeline_notify_events: list[str] = []
pipeline_notify_calls = {"data_frame": 0, "data_calendar": 0}
with tempfile.TemporaryDirectory() as raw_tmp:
    tmp_root = Path(raw_tmp)
    data_notify_config = DataUpdatePipelineConfig(
        name="daily_soxl_data_update",
        dataset_key="daily.soxl_unit",
        dataset_name="unit_soxl_daily",
        universe=("SOXL",),
        output_dir=tmp_root / "data" / "snapshot",
        registry_path=tmp_root / "data" / "data_registry.json",
        calendar_path="data/qlib/soxl/calendars/day.txt",
        source_paths=("provider:unit",),
        start="2026-01-01",
        end="2026-01-05",
        as_of="2026-01-05",
        notify_details=True,
        market_calendar_decision=closed_decision,
        notify_skipped=True,
    )

    def notify_frame_provider(_: DataUpdatePipelineConfig) -> pd.DataFrame:
        pipeline_notify_calls["data_frame"] += 1
        return pd.DataFrame([{"date": "2026-01-02", "instrument": "SOXL", "close": 10.0}])

    def notify_calendar_provider(_: DataUpdatePipelineConfig) -> pd.DatetimeIndex:
        pipeline_notify_calls["data_calendar"] += 1
        return pd.DatetimeIndex(pd.to_datetime(["2026-01-02"]), name="date")

    data_notify_result = run_data_update_once(
        data_notify_config,
        lock_dir=tmp_root / "locks",
        notify=pipeline_notify_events.append,
        frame_provider=notify_frame_provider,
        calendar_provider=notify_calendar_provider,
    )
    data_notify_lock_exists = (tmp_root / "locks" / "daily_soxl_data_update.lock").exists()

decisions_frame = pd.DataFrame(decisions)
decisions_frame.to_csv(OUTPUT_DIR / "market_calendar_decisions.csv", index=False, lineterminator="\n")

summary = {
    "schema_version": "research.schedule_market_calendar_gate.v1",
    "status": "completed",
    "decision_schema_version": decisions[0]["schema_version"],
    "decision_count": int(len(decisions)),
    "market_statuses": sorted(decisions_frame["market_status"].unique().tolist()),
    "closed_status": closed_result.status,
    "closed_skip_reason": closed_result.detail,
    "closed_task_calls": int(calls["count"]),
    "closed_lock_created": bool(closed_lock_exists),
    "notification_events": events,
    "next_trading_date_from_closed": decisions[0]["next_trading_date"],
    "notified_skip_status": notified_skip_result.status,
    "notified_skip_task_calls": int(notification_calls["count"]),
    "notified_skip_lock_created": bool(notified_skip_lock_exists),
    "skip_notification_events": notification_events,
    "pipeline_gate_statuses": {
        "data_update": data_result.status,
        "model_dashboard_refresh": model_result.status,
        "paper_execution": paper_result.status,
    },
    "pipeline_gate_task_calls": pipeline_calls,
    "pipeline_gate_lock_created": bool(pipeline_lock_created),
    "pipeline_gate_artifact_created": bool(pipeline_artifact_created),
    "pipeline_gate_notification_events": pipeline_events,
    "pipeline_skip_notification_status": data_notify_result.status,
    "pipeline_skip_notification_task_calls": pipeline_notify_calls,
    "pipeline_skip_notification_lock_created": bool(data_notify_lock_exists),
    "pipeline_skip_notification_events": pipeline_notify_events,
    "pipeline_surfaces_with_gate": ["data_update", "model_dashboard_refresh", "paper_execution"],
    "artifacts": {
        "market_calendar_decisions": "market_calendar_decisions.csv",
        "review": "schedule_market_calendar_gate_review.md",
    },
}
(OUTPUT_DIR / "schedule_market_calendar_gate_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(OUTPUT_DIR / "schedule_market_calendar_gate_review.md").write_text(
    "# Schedule Market Calendar Gate Review\n\n"
    "## 中文结论\n\n"
    "- 做了什么：验证 `src/schedule` 可在 job 与 pipeline 执行前识别 normal、half_day 和 closed 市场日历状态，并验证可选 skipped 通知摘要。\n"
    "- 核心指标：closed job 返回 skipped，task 调用次数为 0，lock_created=False，默认不发送成功/失败通知；notify_skipped=True 时通知包含 market_status、skip_reason 和 next_trading_date。data_update、model_dashboard_refresh 和 paper_execution pipeline 同样在 provider/exporter/输入读取前 skipped。\n"
    "- regime 差异：本卡片不评估市场 regime；它处理的是 schedule 日历状态差异，供数据更新、模型刷新、dashboard export 和 paper/sim 复用。\n"
    "- 是否晋升：否；这是 schedule 框架 gate，不是模型或策略晋升。\n"
    "- 风险：若真实休市或半日市来源缺失，pipeline 可能在错误日期跳过或执行；production notebook 需要固定真实交易日历来源。\n"
    "- 下一步：在生产定时配置中统一注入真实日历决策，并固定真实交易日历、半日市和休市原因来源。\n",
    encoding="utf-8",
)
summary


{'schema_version': 'research.schedule_market_calendar_gate.v1',
 'status': 'completed',
 'decision_schema_version': 'schedule.market_calendar_decision.v1',
 'decision_count': 3,
 'market_statuses': ['closed', 'half_day', 'normal'],
 'closed_status': 'skipped',
 'closed_skip_reason': "New Year's Day",
 'closed_task_calls': 0,
 'closed_lock_created': False,
 'notification_events': [],
 'next_trading_date_from_closed': '2026-01-02',
 'notified_skip_status': 'skipped',
 'notified_skip_task_calls': 0,
 'notified_skip_lock_created': False,
 'skip_notification_events': ["paper_execution:skipped | market_status=closed | skip_reason=New Year's Day | next_trading_date=2026-01-02"],
 'pipeline_gate_statuses': {'data_update': 'skipped',
  'model_dashboard_refresh': 'skipped',
  'paper_execution': 'skipped'},
 'pipeline_gate_task_calls': {'data_frame': 0,
  'data_calendar': 0,
  'matrix': 0,
  'dashboard': 0},
 'pipeline_gate_lock_created': False,
 'pipeline_gate_artifact_created': False,
 'pipelin

## 中文结论

- 做了什么：新增并验证 `src/schedule` 的市场日历 gate，覆盖 normal、half_day 和 closed 三类调度日期，并把 closed gate 透传到 data_update、model_dashboard_refresh 和 paper_execution pipeline；同时验证可选 skipped 通知摘要。
- 核心指标：closed 日期返回 `skipped`，skip reason 记录休市原因，基础 job task 调用次数为 0，不创建 lock，默认不发送成功/失败通知；`notify_skipped=True` 时通知包含 market_status、skip_reason 和 next_trading_date。三个 pipeline 的 provider/exporter/输入读取调用次数也为 0，不产生输出 artifact。
- regime 差异：本 notebook 不分析交易 regime；这里的差异是日历状态差异，用于避免 schedule 在休市或半日市上误触发数据、模型、dashboard 或 paper/sim 流程。
- 是否晋升：否；这是 `src/schedule` 基础契约补强，不是策略晋升。
- 风险：当前示例使用小型静态交易日历；production notebook 需要固定真实交易日历来源、半日市列表和休市原因来源。
- 下一步：在生产定时配置中统一注入真实日历决策，并固定真实交易日历、半日市和休市原因来源。
